<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [11]</a>'.</span>

# E2E - Speech-to-Speech Evaluation Pipeline

This notebook runs an entire speech to speech evaluation pipeline from start to end:

1. Sets up utility functions to manage non-blocking background cells.
2. Starts the React frontend and Python backend of a S2S application ( in our example the Nova Sonic Test Suite application)
3. Starts the Playwright test suite
4. Provide AnnotationUI for mapping sessions to validation data.
5. Runs the evaluation on an entire conversation flow of a session.

## Prerequisites

a. Test data in sub-directories of the [test data directory](./test-data/) for each test case that contains numerical labeled audio data in wav format along with a config file that follows the [config template](./test-data/CONFIG_TEMPLATE.md) format.
Audio will be triggered in order in response to Assistant audio.

b. A defined [validation dataset](./data/s2s_validation_dataset.jsonl)

c. Python Library dependencies


In [1]:
# !pip install seaborn pandas python-dotenv boto3

## Cell 1: Setup imports and utilities

In [2]:
import subprocess
import os
import sys
import time
import asyncio
from pathlib import Path
from threading import Thread
import signal
import requests
import boto3
from datetime import datetime, timedelta, timezone
import json
from dotenv import load_dotenv

from s2s_annotation_ui import AnnotationUI
import s2s_evaluator as eval_module

# ============================================================================
# Initialize AWS Clients
# ============================================================================

# Load environment variables from .env file if it exists
env_file = Path(".env")
if env_file.exists():
    load_dotenv(env_file)
    print(f"✅ Loaded environment variables from {env_file}")
else:
    print(f"⚠️ No .env file found at {env_file}, using system environment variables")


# AWS Configuration
AWS_REGION = os.getenv('AWS_REGION', os.getenv('AWS_DEFAULT_REGION', 'us-east-1'))
AWS_PROFILE = os.getenv('AWS_PROFILE', None)

# Optional: Explicit credentials (not recommended for production, use IAM roles or profiles)
AWS_ACCESS_KEY_ID = os.getenv('AWS_ACCESS_KEY_ID', None)
AWS_SECRET_ACCESS_KEY = os.getenv('AWS_SECRET_ACCESS_KEY', None)
AWS_SESSION_TOKEN = os.getenv('AWS_SESSION_TOKEN', None)

# Initialize boto3 session
session_kwargs = {'region_name': AWS_REGION}
if AWS_PROFILE:
    session_kwargs['profile_name'] = AWS_PROFILE
if AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY:
    session_kwargs['aws_access_key_id'] = AWS_ACCESS_KEY_ID
    session_kwargs['aws_secret_access_key'] = AWS_SECRET_ACCESS_KEY
    if AWS_SESSION_TOKEN:
        session_kwargs['aws_session_token'] = AWS_SESSION_TOKEN

try:
    boto3_session = boto3.Session(**session_kwargs)
    
    # Initialize CloudWatch Logs client
    cloudwatch_logs_client = boto3_session.client('logs')    
    bedrock_runtime_client = boto3_session.client('bedrock-runtime')
    bedrock_client = boto3_session.client('bedrock')

    # Test if credentials are valid
    sts_client = boto3_session.client('sts')
    identity = sts_client.get_caller_identity()  
    # extract last 4 digits of account
    print(f"✅ Initialized AWS clients for account {identity['Account'][-4:]}")


except Exception as e:
    print(f"\n❌ Failed to initialize AWS clients: {e}")
    print(f"   Please check your AWS credentials and configuration")
    raise


# Configuration
EVAL_DATASET_PATH = "data/s2s_eval_data.jsonl"
VALIDATION_DATASET_PATH = "data/s2s_validation_dataset.jsonl"
MAPPINGS_FILE = Path("config/manual_mappings.json")

# Initialize evaluator with existing boto3 session
evaluator = eval_module.S2SEvaluator(boto3_session=boto3_session)

# Store process references globally
processes = {}

def start_background_process(name, command, cwd=None):
    """
    Start a background process and store reference
    """
    try:
        process = subprocess.Popen(
            command,
            cwd=cwd,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            shell=True,
            preexec_fn=os.setsid  # Unix: create new session
        )
        processes[name] = {
            'process': process,
            'cwd': cwd,
            'command': command
        }
        print(f"✅ Started {name} (PID: {process.pid})")
        return process
    except Exception as e:
        print(f"❌ Failed to start {name}: {e}")
        return None

def stop_background_process(name):
    """
    Stop a background process
    """
    if name not in processes:
        print(f"⚠️ Process {name} not found")
        return
    
    process = processes[name]['process']
    if process.poll() is None:  # Process still running
        try:
            os.killpg(os.getpgid(process.pid), signal.SIGTERM)
            process.wait(timeout=5)
            print(f"✅ Stopped {name}")
        except Exception as e:
            print(f"❌ Failed to stop {name}: {e}")
    else:
        print(f"ℹ️ {name} already stopped")
    
    del processes[name]

def check_process_status(name):
    """
    Check if process is running
    """
    if name not in processes:
        return f"❌ {name} not started"
    
    process = processes[name]['process']
    if process.poll() is None:
        return f"✅ {name} is running (PID: {process.pid})"
    else:
        return f"❌ {name} has stopped (exit code: {process.poll()})"

def list_processes():
    """
    List all managed processes
    """
    print("\n=== Running Processes ===")
    if not processes:
        print("No processes running")
    else:
        for name, info in processes.items():
            process = info['process']
            status = "✅ Running" if process.poll() is None else "❌ Stopped"
            print(f"{status} - {name} (PID: {process.pid})")


02:37:01 [INFO] Found credentials in shared credentials file: ~/.aws/credentials


⚠️ No .env file found at .env, using system environment variables


02:37:01 [INFO] Using provided boto3 session


02:37:01 [INFO] ✅ S2SEvaluator initialized


✅ Initialized AWS clients for account 2101


## Cell 2: Start Python Backend (WebSocket Server)

In [3]:
# Start Python backend
backend_dir = Path("sample_s2s_app/python-server").resolve()
print(f"Backend directory: {backend_dir}")
print(f"Backend exists: {backend_dir.exists()}")

if backend_dir.exists():
    # Check if shell script exists
    app_py = backend_dir / "run_server_with_telemetry.sh"
    if app_py.exists():
        # ./run_server_with_telemetry.sh 2>&1 | tee "telemetry.log"
        start_background_process(
            "python-server",
            './run_server_with_telemetry.sh 2>&1 | tee "telemetry.log"',
            cwd=str(backend_dir)
        )
        print(f"Backend server should be running on port 8000 (or configured port)")
        time.sleep(2)  # Give backend time to start
    else:
        print(f"❌ app.py not found in {backend_dir}")
        print(f"Contents: {list(backend_dir.iterdir())}")
else:
    print(f"❌ Backend directory not found: {backend_dir}")

Backend directory: /local/home/lukewma/evals-workshop/04-workload-specific-evaluations/04-05-Speech-to-Speech/sample_s2s_app/python-server
Backend exists: True
✅ Started python-server (PID: 28153)
Backend server should be running on port 8000 (or configured port)


## Cell 3: Start React Frontend (Dev Server)

In [4]:
# Start React frontend
frontend_dir = Path("sample_s2s_app/react-client").resolve()
print(f"Frontend directory: {frontend_dir}")
print(f"Frontend exists: {frontend_dir.exists()}")

if frontend_dir.exists():
    # Check if package.json exists
    package_json = frontend_dir / "package.json"
    if package_json.exists():
        start_background_process(
            "react-client",
            "npm run start",
            cwd=str(frontend_dir)
        )
        print(f"Frontend dev server should be running on http://localhost:3000 (or configured port)")
        time.sleep(3)  # Give frontend time to start
    else:
        print(f"❌ package.json not found in {frontend_dir}")
        print(f"Contents: {list(frontend_dir.iterdir())}")
else:
    print(f"❌ Frontend directory not found: {frontend_dir}")

Frontend directory: /local/home/lukewma/evals-workshop/04-workload-specific-evaluations/04-05-Speech-to-Speech/sample_s2s_app/react-client
Frontend exists: True
✅ Started react-client (PID: 28300)
Frontend dev server should be running on http://localhost:3000 (or configured port)


## Cell 4: Check Process Status

In [5]:
list_processes()
print(f"\nBackend: {check_process_status('python-server')}")
print(f"Frontend: {check_process_status('react-client')}")


=== Running Processes ===
❌ Stopped - python-server (PID: 28153)
❌ Stopped - react-client (PID: 28300)

Backend: ❌ python-server has stopped (exit code: 0)
Frontend: ❌ react-client has stopped (exit code: 127)


## Cell 5: Launch Playwright Test Suite

In [6]:
# Wait for servers to be ready before running tests
def wait_for_server(url, max_retries=30, delay=1):
    """
    Wait for a server to be ready
    """
    for i in range(max_retries):
        try:
            response = requests.head(url, timeout=2)
            print(f"✅ {url} is ready")
            return True
        except:
            print(f"⏳ Waiting for {url}... ({i+1}/{max_retries})")
            time.sleep(delay)
    print(f"❌ {url} did not respond in time")
    return False

# Check if servers are ready
frontend_ready = wait_for_server("http://localhost:3000")

if frontend_ready:
    # Start Playwright tests
    print("\n🎭 Starting Playwright tests...")
    start_background_process(
        "playwright-tests",
        "npm run test:e2e",  # Adjust if your test command is different
        cwd=str(Path(".").resolve())
    )
    print("✅ Playwright tests started in background")
else:
    print("⚠️ Frontend not ready, skipping Playwright tests")

⏳ Waiting for http://localhost:3000... (1/30)


⏳ Waiting for http://localhost:3000... (2/30)


⏳ Waiting for http://localhost:3000... (3/30)


⏳ Waiting for http://localhost:3000... (4/30)


⏳ Waiting for http://localhost:3000... (5/30)


...

⏳ Waiting for http://localhost:3000... (27/30)


⏳ Waiting for http://localhost:3000... (28/30)


⏳ Waiting for http://localhost:3000... (29/30)


⏳ Waiting for http://localhost:3000... (30/30)


❌ http://localhost:3000 did not respond in time
⚠️ Frontend not ready, skipping Playwright tests


## Cell 6: View Playwright Test Logs (Optional)

In [7]:
# Optional: Stream logs from Playwright tests
if 'playwright-tests' in processes:
    process = processes['playwright-tests']['process']
    print("📋 Playwright test output:")
    # This will show output as it comes (non-blocking)
    while process.poll() is None:
        try:
            line = process.stdout.readline().decode().strip()
            if line:
                print(line)
        except:
            break
        time.sleep(0.1)

## Cell 7: Pull all trace data from Amazon CloudWatch and parse it into an eval dataset format

This cell uses the **S2SEvaluator** class which provides a unified interface for:
- Extracting traces from CloudWatch (retrieves ALL events, then filters in Python for better compatibility)
- Processing spans into evaluation datasets
- Running LLM-as-a-Judge evaluations
- Generating evaluation reports

**Note**: For maximum compatibility, the extractor retrieves all CloudWatch events without a filter pattern, then filters for span data in Python. This avoids issues with CloudWatch filter patterns not matching certain JSON structures.

In [8]:
# Extract traces
traces = evaluator.extract_traces_from_cloudwatch(hours_back=24)


eval_data = evaluator.process_and_store_eval_dataset([traces], EVAL_DATASET_PATH)

02:37:36 [INFO] Searching CloudWatch for all sessions from the last 24 hour(s)


02:37:36 [INFO] Time range: 2026-04-11 02:37:36.960424 to 2026-04-12 02:37:36.960424


02:37:36 [INFO] CloudWatch region: us-west-2


02:37:36 [INFO] CloudWatch log group: aws/spans


02:37:37 [INFO] Using filter_log_events API to retrieve spans...


...

02:37:37 [INFO] No more events in this batch


02:37:37 [INFO] Total spans retrieved: 0


02:37:37 [INFO] Session grouping: 0 sessions, 0 spans without session ID


02:37:37 [INFO] Retrieved 0 spans across 0 sessions


02:37:37 [INFO] Saved 0 processed sessions to data/s2s_eval_data.jsonl


## Cell 8: Trace annotation

### UI Details

| Component | Purpose | Implementation |
|-----------|---------|----------------|
| **Session selector** | Choose which session to annotate | `widgets.Dropdown` with session IDs |
| **Turn navigator** | Step through turns with slider or buttons | `widgets.IntSlider` + prev/next buttons |
| **Eval turn view** | Show the user/assistant turn from evaluation data | `widgets.Output` with formatted text |
| **Validation example view** | Show corresponding turn from validation dataset | `widgets.Output` with category info |
| **Category selector** | Assign a validation category | `widgets.Dropdown` with predefined categories |
| **Save button** | Persist the mapping for the current turn | Saves to `config/manual_mappings.json` |
| **Status area** | Confirm save / show errors | `widgets.HTML` with colored messages |

---

### Interaction Logic

1. **When a session is selected**
   - Load its `turns` list from `eval_records`
   - Retrieve the matching validation record (if any) from `val_by_session`
   - Set the turn navigator range accordingly

2. **When the turn slider changes**
   - Render the evaluation turn (user or assistant)
   - Render the matching validation turn if available
   - Pre-populate the Category selector with any previously saved mapping

3. **Saving a mapping**
   - On button click, write a JSON entry `{sessionId, category, timestamp}` to `config/manual_mappings.json`
   - Mappings are loaded automatically and persist across notebook sessions

4. **Persisted file format** (`config/manual_mappings.json`)
   ```json
   [
     {
       "sessionId": "abc123",
       "category": "TechnicalInterview",
       "timestamp": "2025-11-22T10:30:00"
     }
   ]
   ```


**Fallback behavior**: If a turn has no category match, the UI allows reviewers to assign one manually, ensuring no session is left unmapped.

In [9]:
# ============================================================================
# Annotation UI - Interactive Session Mapper
# ============================================================================

# Initialize and display the annotation UI
# This UI allows manual mapping of evaluation sessions to validation categories
# Features:
# - Browse through evaluation sessions
# - View 5 turns at a time with pagination
# - Compare with validation examples by category
# - Save session-to-category mappings

print("Initializing Manual Annotation UI...")
print("=" * 60)

# Create the UI instance with dataset paths from above
ui = AnnotationUI(
    eval_dataset_path=EVAL_DATASET_PATH,
    validation_dataset_path=VALIDATION_DATASET_PATH,
    mappings_file=MAPPINGS_FILE
)

# Display the UI
ui.display()

Initializing Manual Annotation UI...
No records found


## Cell 9: Run Evaluation

In [10]:
# ============================================================================
# Run Evaluation
# ============================================================================

# Check if we have mapped eval data to evaluate
eval_data = evaluator.load_eval_dataset(EVAL_DATASET_PATH)

if not eval_data or len(eval_data) == 0:
    print(f"\n{'='*60}")
    print("⚠️ Cannot Run Evaluation: No Data Available")
    print(f"{'='*60}")
    print("Evaluation requires trace data from CloudWatch.")
    print("")
    print("To proceed:")
    print("  1. Run the S2S application and generate conversations")
    print("  2. Verify telemetry is being sent to CloudWatch")
    print("  3. Re-run Cell 7 to extract traces")
    print("  4. Then run this cell again")
    print(f"{'='*60}\n")
else:
    # Load manual mappings
    manual_mappings = []
    if MAPPINGS_FILE.exists():
        manual_mappings = evaluator.load_manual_mappings(str(MAPPINGS_FILE))
        print(f"✅ Loaded {len(manual_mappings)} manual mappings")
    else:
        print(f"⚠️ No manual mappings file found at {MAPPINGS_FILE}")
        print(f"\n{'='*60}")
        print("ℹ️ Manual Mappings Not Found")
        print(f"{'='*60}")
        print("No manual mappings file exists yet.")
        print("")
        print("To create manual mappings:")
        print("  - Use the Annotation UI (Cell 21) to map sessions to categories/test cases")
        print(f"{'='*60}\n")

    # Load configuration and validation data
    try:
        validation_data = evaluator.load_validation_dataset(VALIDATION_DATASET_PATH)
        config = evaluator.load_config("config/llm_judge_s2s_config.json")
        
        print(f"✅ Loaded {len(validation_data)} validation entries")
        
        # Initialize judge
        judge = evaluator.initialize_judge(config)
        print("✅ LLM Judge initialized")
        
        # Setup output directory
        evaluation_timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        evaluation_output_dir = os.path.join(
            "test-evaluation-results",
            f"evaluation_report_{evaluation_timestamp}"
        )
        os.makedirs(evaluation_output_dir, exist_ok=True)
        print(f"Created output directory: {evaluation_output_dir}")
        
        # Evaluation parameters
        num_runs = 1  # Number of evaluation runs
        delay = 60  # Seconds between runs
        category_filter = None  # Filter by category (None = all)
        
        # Run evaluations
        run_results = []
        for run in range(num_runs):
            print(f"Starting evaluation run {run + 1}/{num_runs}")
            
            try:
                iteration_result = evaluator.run_evaluation_iteration(
                    category_filter=category_filter,
                    eval_data=eval_data,
                    validation_data=validation_data,
                    manual_mappings=manual_mappings
                )
                run_results.append(iteration_result)
                
                # Save run result
                run_file = os.path.join(evaluation_output_dir, f"run_{run + 1}_results.json")
                with open(run_file, 'w') as f:
                    json.dump(iteration_result, f, indent=2)
                print(f"Saved run {run + 1} results to {run_file}")
                
                if run < num_runs - 1:
                    print(f"Waiting {delay} seconds before next run...")
                    time.sleep(delay)
                    
            except Exception as e:
                print(f"Error in evaluation run {run + 1}: {e}")
                print("Continuing with next run...")
                continue
        
        # Merge results
        if run_results:
            print("Merging evaluation results from all runs...")
            merged_results = evaluator.merge_results(run_results)
            
            # Check if any evaluations were performed
            total_evaluations = merged_results.get('total_evaluations', 0)
            
            if total_evaluations == 0:
                print("No matching records found between validation dataset and evaluation dataset")
                print(f"\n{'='*60}")
                print("⚠️ No Matching Sessions")
                print(f"{'='*60}")
                print("No sessions from CloudWatch could be matched with validation dataset.")
                print("")
                print("Possible reasons:")
                print("  1. The test scenarios haven't been run yet")
                print("  2. No manual mappings created yet")
                print("")
                print("Try:")
                print("  - Run the S2S application to generate test sessions")
                print("  - Use the Manual Annotation UI (Cell 21) to map sessions")
                print("  - Verify validation dataset has correct categories")
                print(f"{'='*60}\n")
            else:
                # Save merged results
                merged_file = os.path.join(evaluation_output_dir, "complete_results.json")
                with open(merged_file, 'w') as f:
                    json.dump(merged_results, f, indent=2)
                print(f"Saved merged results to {merged_file}")
                
                # Generate report
                print("Generating evaluation report...")
                report = evaluator.generate_evaluation_report(merged_results)
                report_file = os.path.join(evaluation_output_dir, "evaluation_report.md")
                with open(report_file, 'w') as f:
                    f.write(report)
                print(f"✅ Saved report to {report_file}")
                
                # Display summary
                print(f"\n{'='*60}")
                print(f"📊 Evaluation Complete")
                print(f"{'='*60}")
                print(f"Total evaluations: {total_evaluations}")
                print(f"Number of runs: {num_runs}")
                print(f"Manual mappings used: {len(manual_mappings)}")
                print(f"Report location: {report_file}")
                print(f"{'='*60}\n")
        else:
            print("No evaluation runs completed successfully")
            
    except Exception as e:
        print(f"Failed to run evaluation: {e}")
        print("Please check configuration files and validation dataset")
        raise

02:37:37 [INFO] Loaded eval dataset from data/s2s_eval_data.jsonl with 0 entries



⚠️ Cannot Run Evaluation: No Data Available
Evaluation requires trace data from CloudWatch.

To proceed:
  1. Run the S2S application and generate conversations
  2. Verify telemetry is being sent to CloudWatch
  3. Re-run Cell 7 to extract traces
  4. Then run this cell again



## Cell 10: Visualize Evalation Results

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [11]:
evaluator.visualize_evaluation_results(merged_results=merged_results,  evaluation_output_dir=evaluation_output_dir)

NameError: name 'merged_results' is not defined

## Cell 11: Stop All Processes

In [ ]:
# Stop all processes
print("Stopping all processes...\n")

for name in list(processes.keys()):
    stop_background_process(name)
    time.sleep(0.5)

print("\n✅ All processes stopped")